In [1]:
# standard libraries 
import numpy as np 
import pandas as pd 
import os
import json
import re
from datetime import datetime, date, time
import matplotlib.pyplot as plt

# batch processing: garbage collection and time
import gc
from tqdm.notebook import tqdm
import requests


#image processing
import PIL.Image, PIL.ImageOps 
from IPython.display import display
import cv2 as cv 
import ast 
import tempfile


# loads fathomnet files and train_metadata from eda
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/augmentation/utils.py
/kaggle/input/train-metadata/train_joined_metadata.csv
/kaggle/input/fathomnet-2025/dataset_test.json
/kaggle/input/fathomnet-2025/requirements.txt
/kaggle/input/fathomnet-2025/download.py
/kaggle/input/fathomnet-2025/dataset_train.json


In [4]:
import sys
sys.path.insert(1, "/kaggle/input/augmentation")

import utils  
print("augmentation py file ready!")

augmentation py file ready!


Pipeline to generate the following datasets: 

1. **Baseline- cropped**
- Images cropped to bounding box size
- No augmentation
- Gray scale
- CLAHE (histogram equalization)
- Gaussian normalization

2. **Augmented 1 - cropped and resized**
- train_joined_metadata_aug1.csv
- Images cropped to bounding box size
- Flipped images
- Gray scale
- CLAHE (histogram equalization)
- Gaussian normalization
- Resize image to bioclip input size (224 x 224 pixels)

3. **Augmented 2 - cropped and padded**
- train_joined_metadata_aug1.csv
- Images cropped to bounding box size
- Flipped images
- Gray scale
- CLAHE (histogram equalization)
- Gaussian normalization
- Pad image to bioclip input size (224 x 224 pixels)


In [ ]:
# !rm -rf /kaggle/working/*

In [2]:
# load train_metadata
train_metadata = pd.read_csv("/kaggle/input/train-metadata/train_joined_metadata.csv")
train_metadata.head()

,id_annotation_image_id,image_id,category_id,area,bbox,width,height,file_name,coco_url,date_captured,name
0,1,1,71,943.0,"[491.0, 254.0, 23.0, 41.0]",720,368,67cda248-6d28-4801-9c4e-e4525189ea38.png,https://database.fathomnet.org/static/m3/frame...,2008-12-19 19:46:56,Sebastolobus
1,2,2,8,71577.0,"[701.0, 505.0, 241.0, 297.0]",1920,1080,d9b399f3-8628-4138-a339-f4520be751c5.png,https://database.fathomnet.org/static/m3/frame...,2019-05-30 16:41:04,Apostichopus leukothele
2,3,3,67,533148.0,"[458.0, 332.0, 924.0, 577.0]",1920,1080,3289c3e1-40f2-4512-992f-3fa406b50a86.png,https://database.fathomnet.org/static/m3/frame...,2014-06-05 20:04:19,Scotoplanes
3,4,4,37,299186.0,"[1155.0, 71.0, 659.0, 454.0]",1920,1080,801dc37d-1aac-49ac-9067-93ae4bb7c8b6.png,https://database.fathomnet.org/static/m3/frame...,2017-12-21 15:37:28,Keratoisis
4,5,4,37,520650.0,"[66.0, 6.0, 975.0, 534.0]",1920,1080,801dc37d-1aac-49ac-9067-93ae4bb7c8b6.png,https://database.fathomnet.org/static/m3/frame...,2017-12-21 15:37:28,Keratoisis


In [5]:
#Create an instance of the bioclippreprocessing class from the module
processor = utils.BioClipPreprocessor(
    metadata_path='/kaggle/input/train-metadata/train_joined_metadata.csv',
    base_output_dir='/kaggle/working/processed'
)

# Process data in batches 

In [6]:
def process_dataset(processor, metadata_path, batch_size=32):
    """
    Download images from URLs, process them using multiple pipelines,
    and add new columns for the generated files.
    
    Args:
        processor: BioClipPreprocessor instance
        metadata_path: Path to train_joined_metadata.csv
        batch_size: Number of images to process at once
    
    Returns:
        Updated metadata DataFrame with new columns
    """
    # Load metadata
    metadata_df = pd.read_csv(metadata_path)
    print(f"Loaded metadata with {len(metadata_df)} rows")
    
    # Create temp directory for downloaded images
    download_dir = '/kaggle/working/downloaded_images'
    os.makedirs(download_dir, exist_ok=True)
    
    # Add new columns for processed files
    metadata_df['baseline_file'] = ''
    metadata_df['aug1_file'] = ''
    metadata_df['aug2_file'] = ''
    
    # Process in batches
    for batch_start in tqdm(range(0, len(metadata_df), batch_size)):
        batch_end = min(batch_start + batch_size, len(metadata_df))
        batch_df = metadata_df.iloc[batch_start:batch_end]
        
        # Download images from URLs
        image_paths = []
        for idx, row in batch_df.iterrows():
            try:
                # Get URL from metadata
                url = row['coco_url']
                filename = row['file_name']
                save_path = os.path.join(download_dir, filename)
                
                # Download the image
                response = requests.get(url, stream=True)
                response.raise_for_status()  # Raise an error for bad responses
                
                with open(save_path, 'wb') as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        file.write(chunk)
                
                image_paths.append(save_path)
                print(f"Downloaded: {filename}")
                
            except Exception as e:
                print(f"Error downloading {row.get('file_name', 'unknown')}: {str(e)}")
                continue
        
        # Process with each pipeline
        try:
            # Baseline pipeline
            processor._process_batch(image_paths, 'baseline')
            # Add baseline filenames to metadata
            for i, (_, row) in enumerate(batch_df.iterrows()):
                if i < len(image_paths):  # Only process successfully downloaded images
                    filename = os.path.basename(row['file_name'])
                    metadata_df.loc[batch_start + i, 'baseline_file'] = f"baseline/gauss_clahe_gray_cropped_{filename}"
            
            # Aug1 pipeline
            processor._process_batch(image_paths, 'aug1')
            # Add aug1 filenames to metadata
            for i, (_, row) in enumerate(batch_df.iterrows()):
                if i < len(image_paths):
                    filename = os.path.basename(row['file_name'])
                    metadata_df.loc[batch_start + i, 'aug1_file'] = f"aug1/resized_gauss_clahe_gray_augmented_cropped_{filename}"
            
            # Aug2 pipeline
            processor._process_batch(image_paths, 'aug2')
            # Add aug2 filenames to metadata
            for i, (_, row) in enumerate(batch_df.iterrows()):
                if i < len(image_paths):
                    filename = os.path.basename(row['file_name'])
                    metadata_df.loc[batch_start + i, 'aug2_file'] = f"aug2/padded_gauss_clahe_gray_augmented_cropped_{filename}"
                
        except Exception as e:
            print(f"Error processing batch {batch_start}-{batch_end}: {e}")
        
        # Clean up downloaded images to save space
        for img_path in image_paths:
            try:
                os.remove(img_path)
            except:
                pass
            
        # Clean up after batch
        gc.collect()
        
        # Save checkpoint after each batch
        if batch_start % (batch_size * 10) == 0 and batch_start > 0:
            metadata_df.to_csv('/kaggle/working/updated_metadata_checkpoint.csv', index=False)
            print(f"Saved checkpoint at batch {batch_start}")
    
    # Save final result
    output_path = '/kaggle/working/train_joined_metadata_processed.csv'
    metadata_df.to_csv(output_path, index=False)
    print(f"Processing complete. Updated metadata saved to {output_path}")
    
    return metadata_df

Run `process_dataset` function

In [7]:
updated_metadata = process_dataset(
    processor=processor,
    metadata_path='/kaggle/input/train-metadata/train_joined_metadata.csv',
    batch_size=4
)

Loaded metadata with 23699 rows


  0%|          | 0/5925 [00:00<?, ?it/s]

Downloaded: 67cda248-6d28-4801-9c4e-e4525189ea38.png
Downloaded: d9b399f3-8628-4138-a339-f4520be751c5.png
Downloaded: 3289c3e1-40f2-4512-992f-3fa406b50a86.png
Downloaded: 801dc37d-1aac-49ac-9067-93ae4bb7c8b6.png
Downloaded: 801dc37d-1aac-49ac-9067-93ae4bb7c8b6.png
Downloaded: 0026b62d-3ebc-4a70-b62e-5767a4e5f8b8.png
Downloaded: f0d3c132-de35-470e-8623-36848711daee.png
Downloaded: 32209bb5-4fea-4379-9484-85c5cdb24e9a.png
Downloaded: 32209bb5-4fea-4379-9484-85c5cdb24e9a.png
Downloaded: 32209bb5-4fea-4379-9484-85c5cdb24e9a.png
Downloaded: 644c9d2f-680f-46c8-8c17-99ca930bb505.png
Downloaded: a59f1195-5406-444f-871d-276fd62f554b.png
Downloaded: a59f1195-5406-444f-871d-276fd62f554b.png
Downloaded: a59f1195-5406-444f-871d-276fd62f554b.png
Downloaded: a59f1195-5406-444f-871d-276fd62f554b.png
Downloaded: a59f1195-5406-444f-871d-276fd62f554b.png
Downloaded: a59f1195-5406-444f-871d-276fd62f554b.png
Downloaded: a59f1195-5406-444f-871d-276fd62f554b.png
Downloaded: a59f1195-5406-444f-871d-276fd62f55

KeyboardInterrupt: 

In [ ]:
#PSEUDOCODE
# have a utils.py file of all functions, and call functions from them.
# import functions that you need
# have experiments 
# make sure that input and output make sense
# write into doc string. 
# add a prefrix for either baseline, aug1, aug2
# def download_images(url, filename):
#     '''Download as an image based on the metadata url'''
#     os.makedirs(IMAGE_DIR, exist_ok=True)
#     path = os.path.join(IMAGE_DIR, filename)
#     response = requests.get(url, stream=True)
#     response.raise_for_status()
#     with open(path, 'wb') as out_file:
#         shutil.copyfileobj(response.raw, out_file)
#     return path

# def crop_images(image_paths, train_metadata, output_dir):
#     """
#     Crop all images using their bounding box metadata and save results.
    
#     Parameters:
#         image_paths (list): Paths to original images
#         train_metadata (pd.DataFrame): DataFrame containing metadata
#         output_dir (str): Directory to save cropped images
#     """
#     # Create output directory if needed
#     os.makedirs(output_dir, exist_ok=True)
    
#     for idx, img_path in enumerate(image_paths):
#         try:
#             # Get metadata for current image
#             metadata = train_metadata.iloc[idx]
#             bbox = ast.literal_eval(metadata['bbox'])
#             img_width = metadata['width']
#             img_height = metadata['height']
            
#             # Calculate coordinates
#             x, y, w, h = map(int, bbox)
#             right = x + w
#             lower = y + h
            
#             # Clamp coordinates to image boundaries
#             left = max(0, x)
#             upper = max(0, y)
#             right = min(right, img_width)
#             lower = min(lower, img_height)
            
#             # Validate coordinates
#             if right <= left or lower <= upper:
#                 raise ValueError(f"Invalid bbox {left, upper, right, lower}")
            
#             # Crop and save
#             with Image.open(img_path) as img:
#                 cropped = img.crop((left, upper, right, lower))
#                 output_path = os.path.join(output_dir, f"cropped_{os.path.basename(img_path)}")
#                 cropped.save(output_path)
                
#             print(f"Successfully processed: {img_path}")
            
#         except Exception as e:
#             print(f"Error cropping {img_path}: {str(e)}")
#             continue

# def augment(image_paths, train_metadata, output_dir)
#     '''Rotates or flip images'''
#     os.makedirs(output_dir, exist_ok=True)
    
#     for idx, img_path in enumerate(image_paths):
#         try:
#              with Image.open(img_path) as img:
#                 augmented = tf.image.random_flip_left_right(img)
#                 output_path = os.path.join(output_dir, f"augmented_{os.path.basename(img_path)}")
#                 augmented.save(output_path)
                
#             print(f"Successfully processed: {img_path}")
            
#         except Exception as e:
#             print(f"Error flipping {img_path}: {str(e)}")
#             continue


# def grayscale(image_paths, train_metadata, output_dir)
#     '''Converts image to grayscale'''
#     os.makedirs(output_dir, exist_ok=True)
    
#     for idx, img_path in enumerate(image_paths):
#         try:
#              with Image.open(img_path) as img:
#                 grayscale = tf.image.rgb_to_grayscale(img)/255.0
#                 output_path = os.path.join(output_dir, f"grayscale_{os.path.basename(img_path)}")
#                 grayscale.save(output_path)
                
#             print(f"Successfully processed: {img_path}")
            
#         except Exception as e:
#             print(f"Error grayscaling {img_path}: {str(e)}")
#             continue

# def clahe(image_paths, train_metadata, output_dir)
#     '''Performs histogram equalization on images'''
#      os.makedirs(output_dir, exist_ok=True)
    
#     for idx, img_path in enumerate(image_paths):
#         try:
#              with Image.open(img_path) as img:
#                 clahe = cv2.createCLAHE(clipLimit=5)
#                 clahe_img = clahe.apply(img) + 30
#                 output_path = os.path.join(output_dir, f"clahe_{os.path.basename(img_path)}")
#                 clahe_img.save(output_path)
                
#             print(f"Successfully processed: {img_path}")
            
#         except Exception as e:
#             print(f"Error processing CLAHE {img_path}: {str(e)}")
#             continue


# def gaussian(image_paths, train_metadata, output_dir)
#     '''Performs Gaussian normalization'''
#     for idx, img_path in enumerate(image_paths):
#             try:
#                  with Image.open(img_path) as img:
#                     gaussian = cv2.GaussianBlur(img, (7, 7), 0)
#                     output_path = os.path.join(output_dir, f"gaussian_{os.path.basename(img_path)}")
#                     gaussian.save(output_path)
                    
#                 print(f"Successfully processed: {img_path}")
                
#             except Exception as e:
#                 print(f"Error processing Gaussian blur {img_path}: {str(e)}")
#                 continue

# # resize 
# def resize(image_paths, train_metadata, output_dir, target_size = (224, 224))
#     '''Resizes or pads the image'''
#     for idx, img_path in enumerate(image_paths):
#             try:
#                  with Image.open(img_path) as img:
#                      if size_type = "resize":
#                         image = cv2.resize(img, target_size)


# # pad 
# def pad(image_paths, train_metadata, output_dir, target_size = (224, 224))
#     '''Pads the image'''
#     for idx, img_path in enumerate(image_paths):
#             try:
#                  with Image.open(img_path) as img:
#                      image = cv2.resize(img, target_size)
                    

In [ ]:
# visualize original images
if len(IMAGE_NAMES) != len(IMAGE_URLS):
    raise ValueError("The number of image names must match the number of image URLs.")

image_paths = [download_image(url, name) for url, name in zip(IMAGE_URLS, IMAGE_NAMES)]

print("Image paths:")
print(image_paths)